# Deep Learning Seismic Waveform Detection: 1D-CNN
## Physics-Informed Earthquake vs. Noise Classification

**Project:** Minor Project - Seismic Activity Analysis in North-East India  
**Task:** Classify raw 100 Hz vertical-component (BHZ) seismic recordings (10,000 samples = 100 seconds) as **Earthquake (1)** or **Ambient Noise (0)**.  
**Author / Purpose:** Interactive hyperparameter tuning, evaluation, and presentation notebook.

---
### How to Use This Notebook:
1. **Cell 2 (Configuration):** Modify any hyperparameter (learning rate, batch size, filters, dropout, epochs) to tune the model.
2. **Cell 3 to 7:** Run all cells sequentially to train the model on CUDA GPU / CPU and view the resulting metrics and plots.
3. **Cell 8 (Interactive Predictor):** Test the model on any individual sample and plot the waveform with its predicted probability!

In [ ]:
# ==============================================================================
# STEP 1: IMPORTS & HARDWARE ACCELERATION SETUP
# ==============================================================================
import os
import sys
import time
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report
)

# Device auto-detection (NVIDIA RTX 3050 GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 70)
print(f"Using Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name:     {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"VRAM:         {total_mem_gb:.2f} GB")
print("=" * 70)

---
## ⚙️ Hyperparameter Configuration Panel
**Change any of the values in the cell below to tune and experiment with your model!**

In [ ]:
# ==============================================================================
# STEP 2: HYPERPARAMETER TUNING PANEL (EDIT FREELY!)
# ==============================================================================
CONFIG = {
    # Training Optimization
    "batch_size": 32,            # Mini-batch size (e.g. 16, 32, 64)
    "learning_rate": 0.001,      # Initial learning rate for AdamW (e.g. 0.001, 0.0005)
    "weight_decay": 1e-4,        # L2 regularization to prevent overfitting
    "max_epochs": 25,            # Maximum training epochs
    "early_stop_patience": 5,    # Epochs to wait without validation improvement
    "lr_reduce_patience": 2,     # Epochs before halving the learning rate
    
    # Architecture Tuning
    "conv1_filters": 8,          # Number of feature maps in Block 1
    "conv1_kernel": 9,           # Kernel width in Block 1 (stride=2)
    "conv2_filters": 16,         # Number of feature maps in Block 2
    "conv2_kernel": 7,           # Kernel width in Block 2 (stride=2)
    "conv3_filters": 32,         # Number of feature maps in Block 3
    "conv3_kernel": 5,           # Kernel width in Block 3 (stride=1)
    "dense_hidden": 16,          # Neurons in penultimate dense layer
    "dropout_rate": 0.3,         # Dropout probability for regularization
    
    # Evaluation
    "classification_threshold": 0.5, # Decision threshold for positive class
    "random_seed": 42
}

# Set random seed for reproducibility
torch.manual_seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["random_seed"])

print("Configuration Loaded:")
for k, v in CONFIG.items():
    print(f"  {k:26s} : {v}")

---
## 📂 Data Loading & Dataset Pipeline

In [ ]:
# ==============================================================================
# STEP 3: PATH RESOLUTION & PYTORCH DATASET
# ==============================================================================
def find_dataset_files():
    candidate_dirs = [
        "data wave/synthetic_waveform_simulator/output/phase11_preprocessed_dataset",
        "cnn wave/data wave/synthetic_waveform_simulator/output/phase11_preprocessed_dataset",
        "cnn/cnn wave/data wave/synthetic_waveform_simulator/output/phase11_preprocessed_dataset",
        "../data wave/synthetic_waveform_simulator/output/phase11_preprocessed_dataset",
        "../../cnn wave/data wave/synthetic_waveform_simulator/output/phase11_preprocessed_dataset"
    ]
    for d in candidate_dirs:
        if os.path.isdir(d) and os.path.isfile(os.path.join(d, "train.h5")):
            return {
                "train": os.path.join(d, "train.h5"),
                "val": os.path.join(d, "validation.h5"),
                "test": os.path.join(d, "test.h5")
            }
    raise FileNotFoundError("Phase 11 dataset files (train.h5, validation.h5, test.h5) not found!")

PATHS = find_dataset_files()
print("Found Dataset Files:")
for split, p in PATHS.items():
    size_mb = os.path.getsize(p) / (1024**2)
    print(f"  {split.capitalize():5s}: {p} ({size_mb:.1f} MB)")

class SeismicWaveformDataset(Dataset):
    """Memory-mapped reader for 10,000-sample standardized seismic waveforms."""
    def __init__(self, h5_path, waveform_key="waveforms_standardized"):
        self.h5_path = h5_path
        self.waveform_key = waveform_key
        with h5py.File(h5_path, "r") as f:
            self.length = len(f["labels"])
            self.labels = np.array(f["labels"][:], dtype=np.float32)
        self.f = None

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if self.f is None:
            self.f = h5py.File(self.h5_path, "r", rdcc_nbytes=16*1024*1024)
        wave = self.f[self.waveform_key][idx].astype(np.float32)[np.newaxis, :]
        label = self.labels[idx]
        return torch.from_numpy(wave), torch.tensor([label], dtype=torch.float32)

train_ds = SeismicWaveformDataset(PATHS["train"])
val_ds = SeismicWaveformDataset(PATHS["val"])
test_ds = SeismicWaveformDataset(PATHS["test"])

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)

print(f"\nDataset Loaded Successfully:")
print(f"  Train Samples:      {len(train_ds):,} (Batches: {len(train_loader)})")
print(f"  Validation Samples: {len(val_ds):,} (Batches: {len(val_loader)})")
print(f"  Test Samples:       {len(test_ds):,} (Batches: {len(test_loader)})")

---
## 🧠 Model Architecture: 1D Waveform CNN

In [ ]:
# ==============================================================================
# STEP 4: MODEL DEFINITION
# ==============================================================================
class WaveformCNN(nn.Module):
    """
    1D Convolutional Neural Network for Seismic Event Detection
    Extracts temporal burst features across 10,000 continuous waveform samples.
    """
    def __init__(self, cfg):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: Initial arrival detection
            nn.Conv1d(1, cfg["conv1_filters"], kernel_size=cfg["conv1_kernel"], stride=2, padding=cfg["conv1_kernel"]//2),
            nn.BatchNorm1d(cfg["conv1_filters"]),
            nn.ReLU(),
            nn.MaxPool1d(4),
            
            # Block 2: Intermediate wave packet envelope
            nn.Conv1d(cfg["conv1_filters"], cfg["conv2_filters"], kernel_size=cfg["conv2_kernel"], stride=2, padding=cfg["conv2_kernel"]//2),
            nn.BatchNorm1d(cfg["conv2_filters"]),
            nn.ReLU(),
            nn.MaxPool1d(4),
            
            # Block 3: Coda and dispersion integration
            nn.Conv1d(cfg["conv2_filters"], cfg["conv3_filters"], kernel_size=cfg["conv3_kernel"], stride=1, padding=cfg["conv3_kernel"]//2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(cfg["conv3_filters"], cfg["dense_hidden"]),
            nn.ReLU(),
            nn.Dropout(cfg["dropout_rate"]),
            nn.Linear(cfg["dense_hidden"], 1)
        )

    def forward(self, x):
        feat = self.features(x)
        logits = self.classifier(feat)
        return logits

model = WaveformCNN(CONFIG).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 70)
print(f"Model Architecture: WaveformCNN")
print(f"Total Trainable Parameters: {n_params:,}")
print("=" * 70)
print(model)

---
## 🚀 Training & Validation Loop with Early Stopping

In [ ]:
# ==============================================================================
# STEP 5: TRAINING & VALIDATION LOOP
# ==============================================================================
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=CONFIG["lr_reduce_patience"], verbose=True
)

history = {
    "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": [], "lr": []
}

best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0
best_model_weights = None

print("Starting Model Training...")
print("-" * 70)

start_time = time.time()

for epoch in range(1, CONFIG["max_epochs"] + 1):
    # --- Training Phase ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * len(batch_y)
        preds = (torch.sigmoid(logits) >= CONFIG["classification_threshold"]).float()
        train_correct += (preds == batch_y).sum().item()
        train_total += len(batch_y)
        
    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total
    
    # --- Validation Phase ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            
            val_loss += loss.item() * len(batch_y)
            preds = (torch.sigmoid(logits) >= CONFIG["classification_threshold"]).float()
            val_correct += (preds == batch_y).sum().item()
            val_total += len(batch_y)
            
    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total
    current_lr = optimizer.param_groups[0]["lr"]
    
    history["train_loss"].append(epoch_train_loss)
    history["train_acc"].append(epoch_train_acc)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(epoch_val_acc)
    history["lr"].append(current_lr)
    
    scheduler.step(epoch_val_loss)
    
    # Checkpoint and Early Stopping
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        patience_counter = 0
        best_model_weights = model.state_dict().copy()
        marker = " ★ (Best Checkpoint)"
    else:
        patience_counter += 1
        marker = ""
        
    print(f"Epoch {epoch:2d}/{CONFIG['max_epochs']} | " 
          f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc*100:5.2f}% | " 
          f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc*100:5.2f}% | " 
          f"LR: {current_lr:.1e}{marker}")
          
    if patience_counter >= CONFIG["early_stop_patience"]:
        print(f"\n[EARLY STOP] No validation improvement for {CONFIG['early_stop_patience']} epochs.")
        break

total_sec = time.time() - start_time
print("-" * 70)
print(f"Training Finished in {total_sec:.1f}s. Best Epoch: {best_epoch} (Val Loss: {best_val_loss:.4f})")

# Restore best weights
if best_model_weights is not None:
    model.load_state_dict(best_model_weights)
    print("Restored Best Validation Model Weights.")

---
## 📈 Evaluation on Held-Out Test Set (Unseen Waveforms)

In [ ]:
# ==============================================================================
# STEP 6: TEST SET EVALUATION & QUANTITATIVE METRICS
# ==============================================================================
model.eval()
test_probs, test_targets = [], []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        logits = model(batch_x)
        probs = torch.sigmoid(logits).cpu().numpy().ravel()
        test_probs.extend(probs)
        test_targets.extend(batch_y.numpy().ravel())

test_probs = np.array(test_probs)
test_targets = np.array(test_targets)
test_preds = (test_probs >= CONFIG["classification_threshold"]).astype(int)

acc = accuracy_score(test_targets, test_preds)
prec = precision_score(test_targets, test_preds)
rec = recall_score(test_targets, test_preds)
f1 = f1_score(test_targets, test_preds)
roc_auc = roc_auc_score(test_targets, test_probs)
ap = average_precision_score(test_targets, test_probs)

cm = confusion_matrix(test_targets, test_preds)
tn, fp, fn, tp = cm.ravel()

print("=" * 70)
print(f"HELD-OUT TEST SET RESULTS ({len(test_targets):,} Samples @ Threshold = {CONFIG['classification_threshold']})")
print("=" * 70)
print(f"  Accuracy:          {acc*100:.2f}%")
print(f"  Precision:         {prec*100:.2f}%")
print(f"  Recall:            {rec*100:.2f}% (Detected {tp}/{tp+fn} earthquakes)")
print(f"  F1-Score:          {f1:.4f}")
print(f"  ROC-AUC:           {roc_auc:.6f}")
print(f"  Average Precision: {ap:.6f}")
print("-" * 70)
print(f"Confusion Matrix:")
print(f"  True Negatives (Noise correctly classified):      {tn:,}")
print(f"  False Positives (Noise classified as Earthquake): {fp:,}")
print(f"  False Negatives (Missed Earthquakes):             {fn:,}")
print(f"  True Positives (Detected Earthquakes):            {tp:,}")
print("=" * 70)
print("\nClassification Report:")
print(classification_report(test_targets, test_preds, target_names=["Noise (0)", "Earthquake (1)"], digits=4))

---
## 📊 Visualizations: Learning Curves, Confusion Matrix & ROC/PR

In [ ]:
# ==============================================================================
# STEP 7: COMPREHENSIVE PERFORMANCE PLOTS
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Loss & Accuracy Curves
axes[0].plot(history["train_loss"], label="Train Loss", color="#2196F3", linewidth=2)
axes[0].plot(history["val_loss"], label="Val Loss", color="#F44336", linewidth=2)
axes[0].axvline(best_epoch - 1, color="green", linestyle="--", label=f"Best Epoch ({best_epoch})")
axes[0].set_title("Training & Validation Loss", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch", fontsize=11)
axes[0].set_ylabel("BCE Loss", fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Confusion Matrix Heatmap
im = axes[1].imshow(cm, cmap="Blues", interpolation="nearest")
axes[1].set_title(f"Confusion Matrix (Threshold = {CONFIG['classification_threshold']})", fontsize=13, fontweight="bold")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(["Noise", "Earthquake"], fontsize=11)
axes[1].set_yticklabels(["Noise", "Earthquake"], fontsize=11)
axes[1].set_xlabel("Predicted Label", fontsize=11)
axes[1].set_ylabel("True Label", fontsize=11)
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                     color="white" if cm[i, j] > cm.max()/2 else "black",
                     fontsize=14, fontweight="bold")

# Plot 3: ROC & PR Curves
fpr, tpr, _ = roc_curve(test_targets, test_probs)
prec_vals, rec_vals, _ = precision_recall_curve(test_targets, test_probs)
axes[2].plot(fpr, tpr, label=f"ROC (AUC = {roc_auc:.4f})", color="#4CAF50", linewidth=2)
axes[2].plot(rec_vals, prec_vals, label=f"PR (AP = {ap:.4f})", color="#FF9800", linewidth=2)
axes[2].plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random Chance")
axes[2].set_title("ROC & Precision-Recall Curves", fontsize=13, fontweight="bold")
axes[2].set_xlabel("False Positive Rate / Recall", fontsize=11)
axes[2].set_ylabel("True Positive Rate / Precision", fontsize=11)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 🔍 Interactive Sample Visualizer
Pick any sample index from the test set to plot its raw waveform and see the model's live classification output!

In [ ]:
# ==============================================================================
# STEP 8: INTERACTIVE WAVEFORM INSPECTION
# Change SAMPLE_IDX below to test any trace!
# ==============================================================================
SAMPLE_IDX = 42   # Try any integer from 0 to 2427

waveform, true_label = test_ds[SAMPLE_IDX]
with torch.no_grad():
    prob = torch.sigmoid(model(waveform.unsqueeze(0).to(device))).item()
pred_class = "Earthquake (1)" if prob >= CONFIG["classification_threshold"] else "Noise (0)"
actual_class = "Earthquake (1)" if true_label.item() == 1.0 else "Noise (0)"

time_axis = np.linspace(0, 100, 10000)
plt.figure(figsize=(14, 4))
plt.plot(time_axis, waveform.squeeze(0).numpy(), color="#1565C0", linewidth=0.8)
plt.title(f"Test Sample #{SAMPLE_IDX} | True: {actual_class} | Predicted: {pred_class} (Prob = {prob:.4f})",
          fontsize=12, fontweight="bold",
          color="green" if pred_class == actual_class else "red")
plt.xlabel("Time (seconds)", fontsize=11)
plt.ylabel("Standardized Amplitude", fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()